In [18]:
import requests
import json
import os
import time
from tqdm import tqdm

BASE_DIR = "Scientific_Novelty_Detection_2022_2025"
CACHE_DIR = os.path.join(BASE_DIR, "cache")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

YEAR_START = 2022
YEAR_END = 2025
PER_PAGE = 200
TOP_K = 150

In [19]:
TASK_QUERIES = {
    "Dia": "dialogue systems OR conversational modeling",
    "MT": "machine translation",
    "NLI": "natural language inference",
    "Par": "paraphrase generation",
    "QA": "question answering",
    "SA": "sentiment analysis",
    "Sum": "text summarization"
}

In [20]:
def safe_request(url, retries=5):
    for i in range(retries):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 429:
                time.sleep(2 ** i)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            time.sleep(2 ** i)
    return None

In [21]:
def fetch_skg_task(task, keyword):

    cache_file = os.path.join(CACHE_DIR, f"SKG_{task}_metadata.json")
    checkpoint_file = os.path.join(CHECKPOINT_DIR, f"SKG_{task}_fetch_checkpoint.json")

    if os.path.exists(cache_file):
        print(f"Loading cached SKG for {task}")
        with open(cache_file, "r") as f:
            return json.load(f)

    print(f"\nFetching SKG for {task}")

    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            checkpoint = json.load(f)
        cursor = checkpoint["cursor"]
        papers = checkpoint["papers"]
        print(f"Resuming {task} from cursor {cursor}")
    else:
        cursor = "*"
        papers = []

    while True:

        params = {
            "search": keyword,
            "filter": f"publication_year:{YEAR_START}-{YEAR_END}",
            "sort": "cited_by_count:desc",
            "per-page": PER_PAGE,
            "cursor": cursor,
            "mailto": "your_email@example.com"
        }

        r = requests.get("https://api.openalex.org/works", params=params, timeout=30)

        if r.status_code != 200:
            print("API error:", r.status_code)
            print(r.text)
            break

        data = r.json()
        results = data.get("results", [])

        if not results:
            break

        for paper in results:

            if paper.get("type") not in ["article", "proceedings-article"]:
                continue

            papers.append({
                "id": paper.get("id"),
                "title": paper.get("title"),
                "year": paper.get("publication_year"),
                "cited_by_count": paper.get("cited_by_count"),
                "pdf_url": paper.get("open_access", {}).get("oa_url")
            })

        print(f"{task} — Collected {len(papers)}")

        # Save checkpoint
        with open(checkpoint_file, "w") as f:
            json.dump({
                "cursor": data["meta"]["next_cursor"],
                "papers": papers
            }, f, indent=2)

        if len(papers) >= TOP_K:
            break

        cursor = data["meta"]["next_cursor"]

        if not cursor:
            break

        time.sleep(1)

    papers = papers[:TOP_K]

    with open(cache_file, "w") as f:
        json.dump(papers, f, indent=2)

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)

    print(f"{task} SKG final count: {len(papers)}")

    return papers

In [22]:
all_skg_metadata = {}

for task, query in TASK_QUERIES.items():
    metadata = fetch_skg_task(task, query)
    all_skg_metadata[task] = metadata

print("\nAll SKG metadata fetched successfully.")


Fetching SKG for Dia
Dia — Collected 128
Dia — Collected 252
Dia SKG final count: 150

Fetching SKG for MT
MT — Collected 114
MT — Collected 239
MT SKG final count: 150

Fetching SKG for NLI
NLI — Collected 140
NLI — Collected 278
NLI SKG final count: 150

Fetching SKG for Par
Par — Collected 153
Par SKG final count: 150

Fetching SKG for QA
QA — Collected 122
QA — Collected 255
QA SKG final count: 150

Fetching SKG for SA
SA — Collected 127
SA — Collected 267
SA SKG final count: 150

Fetching SKG for Sum
Sum — Collected 125
Sum — Collected 263
Sum SKG final count: 150

All SKG metadata fetched successfully.
